# --------------------------------------Introduction--------------------------------------
>`Pipeline` is a simple way to `keep data preprocessing and modeling code organized`. Specifically, a `pipeline` `bundles preprocessing and modeling steps` so we can use the whole bundle as if it were a single step.

>Instead of writing separate lines to `fit an imputer, transform train, transform valid, fit an encoder, transform again, concat, drop columns...` we define the whole sequence once, and then call `.fit()` and `.predict()` on the pipeline itself — it runs every step internally, in order, automatically.

>We use Pipeline when have `a series of steps that need to happen in order`, `each one operating on the output of the previous one`, `on the same set of columns`.

>Note =>inside a `Pipeline/ColumnTransformer`, we never manually touch the `output array`

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder # last time it worked well for mrlbourne dataset

In [2]:
file_path="melb_data.csv"
data=pd.read_csv(file_path)

data.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='object')

In [3]:
data.dropna(subset=["Price"],axis=0,inplace=True)

y=data.Price
X=data.drop(["Price"],axis=1)

train_X,valid_X,train_y,valid_y=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=0)

numeric_cols=[]
for col in train_X.columns:
    if train_X[col].dtype in ["int64","float64"]:
        numeric_cols.append(col)

low_cardinality_cols=[]
for col in train_X.columns:
    if train_X[col].dtype=="object" and train_X[col].nunique()<10:
        low_cardinality_cols.append(col)

my_cols=numeric_cols+low_cardinality_cols
train_X=train_X[my_cols].copy()
valid_X=valid_X[my_cols].copy()


print(low_cardinality_cols)

['Type', 'Method', 'Regionname']


# Construct the full pipeline in three steps.

## Step 1: Define Preprocessing Steps
>use the ColumnTransformer class to `bundle together different preprocessing steps`

>Different columns needing different treatment → `ColumnTransformer`

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numerical_transformer=SimpleImputer(strategy="constant")

categorical_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

categorical_cols=[]
for col in train_X.columns:
    if train_X[col].dtype=="object":
        categorical_cols.append(col)

preprocessor=ColumnTransformer(transformers=[
    ("num",numerical_transformer,numeric_cols),
    ("cat",categorical_transformer,categorical_cols)
])

## Step 2: Define the Model
>we define a random forest model with the familiar `RandomForestRegressor` class.

In [5]:
model=RandomForestRegressor(n_estimators=100,random_state=0)

## Step 3: Create and Evaluate the Pipeline

>we use the `Pipeline` that bundles the preprocessing and modeling steps. There are a few important things to notice:

* With the pipeline, we preprocess the training data and fit the model in a single line of code. `(In contrast, without a pipeline, we have to do imputation, one-hot encoding, and model training in separate steps. This becomes especially messy if we have to deal with both numerical and categorical variables!)`

* With the pipeline, we supply the `unprocessed features` in `X_valid` to the `predict()` command, and the pipeline automatically preprocesses the features before generating predictions. (However, without a pipeline, we have to remember to preprocess the validation data before making predictions.)

In [6]:
# # Bundle preprocessing and modeling code in a pipeline
my_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',model)
])

my_pipeline.fit(train_X,train_y)

model_prediction=my_pipeline.predict(valid_X)

error=mean_absolute_error(valid_y,model_prediction)

print('Mean Absolute Error: ',error)

Mean Absolute Error:  160679.18917034855


## Conclusion

>Pipelines are valuable for cleaning up machine learning code and avoiding errors, and are especially useful for workflows with sophisticated data preprocessing.